# TTM-Small-4M — T4 A2 | Huấn luyện 1.024 bước liên tục

**Colab Free GPU ($0)**. Bấm **Runtime → Run all**. Một lượt TRAIN 1.024 bước, **không dừng ở 256**; checkpoint tự động ở 256, 512, 768, 1.024. Có log mỗi 32 bước. Chỉ dùng TRAIN và VAL, tuyệt đối không đọc TEST.

Nếu phiên Colab ngắt: chạy lại **chính notebook này**, nó xác minh checkpoint READY và tiếp tục từ mốc gần nhất trong **cùng RunID**. Nếu đã hoàn tất bước 1.024, không chạy lại. Nguồn mã được nhúng trong notebook theo commit GitHub đã khóa và kiểm SHA-256; không cần truy cập repository trong Colab. Số 55/90 phút cũ **không còn là điều kiện dừng**, nhưng Colab có thể tự thu hồi GPU. Không dùng Colab trả phí.

Thư mục dữ liệu Drive đã xác minh: `MyDrive/TTM/ttm-small-pilot/t4_private_pack_20261108_v1`. Kết quả mới: `MyDrive/TTM/ttm-small-pilot/t4_science_4m_1024_a2_20261108_v1`.

**Chỉ chạy sau khi muốn bắt đầu lượt khoa học 1.024 bước này.** Khi chạy, bước đầu tiên ghi khóa nghiên cứu một lần vào thư mục Drive riêng tư; không thêm vòng xác nhận kỹ thuật.


In [ ]:
# Ô 1 — mã T4 A2 nhúng ngay trong notebook, đối chiếu SHA-256
from pathlib import Path
import hashlib,sys
ROOT=Path('/content/ttm_t4_a2_1024_exact')
ROOT.mkdir(parents=True,exist_ok=True)
SOURCE_COMMIT="314b085c79d2dbb50e2101a29ee59290a70cd586"
FILES={"p02_source_exact.py":["tools/pilot_t4/integrated/p02_source_exact.py","e0fcac285918b66b13c8f694e6d625a0691c9b59221b43043e63f46b25ddcc07"],"t4_capacity_smoke.py":["tools/pilot_t4/integrated/t4_capacity_smoke.py","5030855baad73582f897d3cd128ce24621adb39513bf20f527d875e08ea71411"],"t4_eval.py":["tools/pilot_t4/integrated/t4_eval.py","1502ea5773ab0f7c0869097ecb19b610867d193728d7ce3a78ef82439767ab5b"],"t4_capacity_accept_a2.py":["tools/pilot_t4/a2_continuous1024/t4_capacity_accept_a2.py","8db7ce9111744523a6c2e558a50677e19c56788298a0619eb4cfa924ea6a03c2"],"t4_train_science_a2.py":["tools/pilot_t4/a2_continuous1024/t4_train_science_a2.py","c6298f3381efd4b54e4b3e7aeb8143354f36bb1ed9fd87d08900cee18f7f90e6"]}
EMBEDDED={"p02_source_exact.py":"\"\"\"TTM P02 decoder-only Transformer core. CPU-correctness candidate, not P03 training.\n\nBPE is frozen at 2048 token IDs; this module does not read corpus or held-out TEST.\nThe LayerNorm variant intentionally has the same parameter names and math as P00.\n\"\"\"\nfrom dataclasses import dataclass\nimport math\n\nimport torch\nfrom torch import nn\nfrom torch.nn import functional as F\n\n\n@dataclass(frozen=True)\nclass P02Config:\n    vocab_size: int = 2048\n    max_seq_len: int = 64\n    n_embd: int = 64\n    n_layer: int = 2\n    n_head: int = 4\n    norm_type: str = \"layernorm\"\n\n    def __post_init__(self):\n        if not all(isinstance(v, int) and not isinstance(v, bool) and v > 0\n                   for v in (self.vocab_size, self.max_seq_len, self.n_embd,\n                             self.n_layer, self.n_head)):\n            raise ValueError(\"all model dimensions must be positive integers\")\n        if self.n_embd % self.n_head:\n            raise ValueError(\"n_embd must be divisible by n_head\")\n        if self.norm_type not in (\"layernorm\", \"rmsnorm\"):\n            raise ValueError(\"norm_type must be layernorm or rmsnorm\")\n\n\nclass RMSNorm(nn.Module):\n    def __init__(self, dim: int, eps: float = 1e-5):\n        super().__init__()\n        self.eps = eps\n        self.weight = nn.Parameter(torch.ones(dim))\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        scale = torch.rsqrt(x.square().mean(dim=-1, keepdim=True) + self.eps)\n        return x * scale * self.weight\n\n\ndef _norm(cfg: P02Config) -> nn.Module:\n    if cfg.norm_type == \"layernorm\":\n        return nn.LayerNorm(cfg.n_embd)\n    return RMSNorm(cfg.n_embd)\n\n\nclass CausalSelfAttention(nn.Module):\n    def __init__(self, cfg: P02Config):\n        super().__init__()\n        self.heads = cfg.n_head\n        self.head_dim = cfg.n_embd // cfg.n_head\n        self.qkv = nn.Linear(cfg.n_embd, 3 * cfg.n_embd)\n        self.proj = nn.Linear(cfg.n_embd, cfg.n_embd)\n\n    def _split(self, t: torch.Tensor) -> torch.Tensor:\n        batch, time, channels = t.shape\n        return t.reshape(batch, time, self.heads, self.head_dim).transpose(1, 2)\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        batch, time, channels = x.shape\n        q, k, v = [self._split(a) for a in self.qkv(x).chunk(3, dim=-1)]\n        scores = q @ k.transpose(-2, -1) / math.sqrt(self.head_dim)\n        mask = torch.ones(time, time, dtype=torch.bool, device=x.device).tril()\n        weights = scores.masked_fill(~mask, -torch.inf).softmax(dim=-1)\n        y = weights @ v\n        y = y.transpose(1, 2).contiguous().reshape(batch, time, channels)\n        return self.proj(y)\n\n\nclass Block(nn.Module):\n    def __init__(self, cfg: P02Config):\n        super().__init__()\n        self.ln1 = _norm(cfg)\n        self.attn = CausalSelfAttention(cfg)\n        self.ln2 = _norm(cfg)\n        self.mlp = nn.Sequential(\n            nn.Linear(cfg.n_embd, 4 * cfg.n_embd),\n            nn.GELU(),\n            nn.Linear(4 * cfg.n_embd, cfg.n_embd),\n        )\n\n    def forward(self, x: torch.Tensor) -> torch.Tensor:\n        x = x + self.attn(self.ln1(x))\n        return x + self.mlp(self.ln2(x))\n\n\nclass GPT(nn.Module):\n    \"\"\"P02 core with P00-compatible LayerNorm parameter names/state_dict.\"\"\"\n    def __init__(self, cfg: P02Config):\n        super().__init__()\n        self.cfg = cfg\n        self.tok = nn.Embedding(cfg.vocab_size, cfg.n_embd)\n        self.pos = nn.Embedding(cfg.max_seq_len, cfg.n_embd)\n        self.blocks = nn.ModuleList([Block(cfg) for _ in range(cfg.n_layer)])\n        self.final_ln = _norm(cfg)\n        self.head = nn.Linear(cfg.n_embd, cfg.vocab_size, bias=False)\n        self.head.weight = self.tok.weight\n        self.apply(self._init_weights)\n\n    @staticmethod\n    def _init_weights(module: nn.Module) -> None:\n        if isinstance(module, (nn.Linear, nn.Embedding)):\n            nn.init.normal_(module.weight, mean=0.0, std=0.02)\n            if isinstance(module, nn.Linear) and module.bias is not None:\n                nn.init.zeros_(module.bias)\n\n    def forward(self, tokens: torch.Tensor, targets: torch.Tensor | None = None):\n        if tokens.dtype != torch.long or tokens.ndim != 2:\n            raise ValueError(\"tokens must be LongTensor [batch, time]\")\n        if tokens.shape[0] < 1 or not (1 <= tokens.shape[1] <= self.cfg.max_seq_len):\n            raise ValueError(\"tokens must have nonempty batch/time within max_seq_len\")\n        if torch.any(tokens < 0).item() or torch.any(tokens >= self.cfg.vocab_size).item():\n            raise ValueError(\"token ID outside P02 vocabulary\")\n        if targets is not None and (targets.dtype != torch.long or targets.shape != tokens.shape):\n            raise ValueError(\"targets must match tokens shape and dtype\")\n        time = tokens.shape[1]\n        positions = torch.arange(time, device=tokens.device)\n        x = self.tok(tokens) + self.pos(positions)[None, :, :]\n        for block in self.blocks:\n            x = block(x)\n        logits = self.head(self.final_ln(x))\n        loss = None\n        if targets is not None:\n            loss = F.cross_entropy(logits.reshape(-1, logits.shape[-1]), targets.reshape(-1))\n        return logits, loss\n","t4_capacity_smoke.py":"#!/usr/bin/env python3\n\"\"\"Zero-science TTM T4 capacity benchmark: exact P02 and 4096 scored targets/step.\n\nNo TinyStories, no TEST or model training. Does not authorize science.\nSeparate single-run no-overwrite report, calibrated overhead reserve.\n\"\"\"\nfrom __future__ import annotations\nimport argparse, hashlib, importlib.util, json, math, os, random, sys, time\nfrom pathlib import Path\n\nimport torch\nfrom torch.nn import functional as F\n\nCONFIG=dict(vocab_size=2048,max_seq_len=256,n_embd=256,n_layer=4,n_head=4,norm_type='layernorm')\nP02_BLOB='9762343e06cf800b50574c2871ae1ed58537fb18'\nSEED=20261108\nTOTAL_STEPS=1024\nTOKENS_PER_STEP=4096\nTIME_CAP_SECONDS=5400\n# 20% of total time reserved for data moves, evals, checkpoints, session variability.\nRESERVE_FRACTION=0.25\nVRAM_FRACTION_CAP=0.80\n\n\ndef sha_file(p):\n    h=hashlib.sha256()\n    with Path(p).open('rb') as f:\n        for data in iter(lambda:f.read(1<<20),b''):h.update(data)\n    return h.hexdigest()\n\n\ndef p02_module(path):\n    p=Path(path)\n    b=p.read_bytes()\n    sha1=hashlib.sha1(b'blob '+str(len(b)).encode('ascii')+b'\\0'+b).hexdigest()\n    if sha1!=P02_BLOB:raise RuntimeError('WRONG_P02_GIT_BLOB:'+sha1)\n    spec=importlib.util.spec_from_file_location('t4_p02_exact',p)\n    mod=importlib.util.module_from_spec(spec)\n    sys.modules[spec.name]=mod\n    spec.loader.exec_module(mod)\n    return mod\n\n\ndef device_setup():\n    if not torch.cuda.is_available():raise RuntimeError('T4_CAPACITY_CUDA_REQUIRED_NO_CPU_SUBSTITUTE')\n    gpu=torch.cuda.get_device_properties(0)\n    torch.set_num_threads(2)\n    return gpu\n\n\ndef amp_scaler(mode):\n    if mode=='fp16':return torch.amp.GradScaler('cuda',enabled=True)\n    return None\n\n\ndef single_trial(mod,*,microbatch:int,precision:str,warmup:int,timed:int):\n    assert microbatch in (1,2,4,8)\n    assert (TOKENS_PER_STEP//256)%microbatch==0\n    assert precision in ('fp32','fp16')\n    torch.manual_seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    random.seed(SEED)\n    model=mod.GPT(mod.P02Config(**CONFIG)).cuda()\n    if sum(p.numel() for p in model.parameters())!=3749376:raise RuntimeError('P02_PARAMETERS_CHANGED')\n    opt=torch.optim.AdamW(model.parameters(),lr=3e-4,betas=(.9,.95),weight_decay=.1)\n    scaler=amp_scaler(precision)\n    generator=torch.Generator(device='cpu').manual_seed(SEED+101)\n    accum=(TOKENS_PER_STEP//256)//microbatch\n    peak_alloc=[];elapsed=[];loss_last=None\n    torch.cuda.empty_cache()\n    torch.cuda.reset_peak_memory_stats()\n    model.train()\n    for step in range(warmup+timed):\n        torch.cuda.synchronize();before=time.perf_counter()\n        opt.zero_grad(set_to_none=True)\n        for _ in range(accum):\n            # EXACT 4096 next-token scored targets, independent synthetic token IDs.\n            ids=torch.randint(0,2048,(microbatch,257),dtype=torch.long,generator=generator)\n            x,y=ids[:,:-1].to('cuda',non_blocking=False),ids[:,1:].to('cuda',non_blocking=False)\n            with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=(precision=='fp16')):\n                logits,_=model(x)\n                loss=F.cross_entropy(logits.reshape(-1,2048).float(),y.reshape(-1),reduction='sum')/TOKENS_PER_STEP\n            if not torch.isfinite(loss).item():raise RuntimeError('NONFINITE_BENCHMARK_LOSS')\n            if scaler:scaler.scale(loss).backward()\n            else:loss.backward()\n            loss_last=float(loss.detach())\n        if scaler:\n            scaler.unscale_(opt)\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)\n            scale_before=scaler.get_scale()\n            scaler.step(opt);scaler.update()\n            if scaler.get_scale()<scale_before:raise RuntimeError('FP16_GRADIENT_OVERFLOW_NOT_CAPACITY_PASS')\n        else:\n            torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)\n            opt.step()\n        torch.cuda.synchronize()\n        sec=time.perf_counter()-before\n        if step>=warmup:\n            elapsed.append(sec)\n            peak_alloc.append(torch.cuda.max_memory_reserved())\n    del model,opt,scaler\n    torch.cuda.synchronize()\n    return {\n       'microbatch_windows':microbatch,'context':256,'precision':precision,\n       'gradient_accumulation':accum,'scored_target_tokens_per_optimizer_step':TOKENS_PER_STEP,\n       'warmup_steps':warmup,'timed_steps':timed,\n       'sample_seconds':[round(x,5) for x in elapsed],\n       'median_seconds':sorted(elapsed)[len(elapsed)//2] if len(elapsed)%2 else (sorted(elapsed)[len(elapsed)//2-1]+sorted(elapsed)[len(elapsed)//2])/2,\n       'worst_seconds':max(elapsed),'peak_cuda_reserved_bytes':max(peak_alloc),\n       'last_synthetic_loss':loss_last,\n    }\n\n\ndef candidate_decision(measurements,bytes_total):\n    for m in measurements:\n        # Conservative max step wall clock, including optimizer and 4096 scored targets.\n        m['projected_science_seconds_with_25pct_reserved']=round(TOTAL_STEPS*m['worst_seconds']/(1-RESERVE_FRACTION),2)\n        m['memory_reserved_ratio']=m['peak_cuda_reserved_bytes']/bytes_total\n        m['feasible_on_measured_smoke']=bool(m['projected_science_seconds_with_25pct_reserved']<=TIME_CAP_SECONDS and m['memory_reserved_ratio']<=VRAM_FRACTION_CAP)\n    eligible=[m for m in measurements if m['feasible_on_measured_smoke']]\n    if not eligible:return None\n    # Maximum margin, choose fastest measured worst case; tie by smaller memory and microbatch.\n    return min(eligible,key=lambda x:(x['worst_seconds'],x['peak_cuda_reserved_bytes'],x['microbatch_windows']))\n\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument('--p02',required=True)\n    ap.add_argument('--out',required=True)\n    ap.add_argument('--warmup',type=int,default=2)\n    ap.add_argument('--timed',type=int,default=4)\n    args=ap.parse_args()\n    if args.warmup!=2 or args.timed!=4:raise RuntimeError('BENCH_DURATION_FROZEN')\n    dst=Path(args.out)\n    if dst.exists():raise FileExistsError('CAPACITY_REPORT_EXISTS_ONE_SHOT_NO_RERUN')\n    if dst.parent.is_symlink():raise RuntimeError('OUT_DIR_SYMLINK')\n    dst.parent.mkdir(parents=True,exist_ok=True)\n    mod=p02_module(args.p02)\n    gpu=device_setup()\n    records=[];failures=[]\n    # Fixed finite candidate order. OOM is a stopped candidate, not hidden.\n    for mode,batch in [('fp32',2),('fp16',2),('fp16',4),('fp16',8)]:\n        try:\n            m=single_trial(mod,microbatch=batch,precision=mode,warmup=args.warmup,timed=args.timed)\n            records.append(m)\n            print(f'CAPACITY_CASE={mode}/mb{batch} WORST_S={m[\"worst_seconds\"]:.3f} VRAM_GiB={m[\"peak_cuda_reserved_bytes\"]/(1024**3):.3f}',flush=True)\n        except (torch.cuda.OutOfMemoryError,RuntimeError) as e:\n            failures.append({'precision':mode,'microbatch_windows':batch,'error':str(e)[:240]})\n            print(f'CAPACITY_CASE_FAIL={mode}/mb{batch}: {type(e).__name__}: {str(e)[:140]}',flush=True)\n            torch.cuda.empty_cache()\n    selected=candidate_decision(records,gpu.total_memory)\n    result={'schema':'ttm.t4.capacity.synthetic.v1','status':'PASS_PROVISIONAL_CAPACITY_ONLY' if selected else 'CAPACITY_HOLD',\n       'science_executed':False,'synthetic_only':True,'gpu':gpu.name,'cuda':str(torch.version.cuda),\n       'torch':str(torch.__version__),'p02_git_blob':P02_BLOB,'p02_sha256':sha_file(args.p02),\n       'config':CONFIG,'unique_parameters':3749376,'steps_target':TOTAL_STEPS,'tokens_per_step':TOKENS_PER_STEP,\n       'time_cap_seconds':TIME_CAP_SECONDS,'reserve_fraction':RESERVE_FRACTION,\n       'vram_reserved_limit_fraction':VRAM_FRACTION_CAP,'total_device_memory_bytes':gpu.total_memory,\n       'records':records,'failed_candidates':failures,\n       'selected_technical_candidate':None if selected is None else {\n          'precision':selected['precision'],'microbatch_windows':selected['microbatch_windows'],\n          'gradient_accumulation':selected['gradient_accumulation'],\n          'worst_seconds':selected['worst_seconds'],\n          'projected_science_seconds_with_25pct_reserved':selected['projected_science_seconds_with_25pct_reserved']},\n       'important_limitations':['only synthetic uniform 256-token windows','actual TRAIN mixture of short/long windows could require more microbatches',\n          'VAL, checkpoint/Drive IO and downtime not directly measured; 25 pct reserve heuristic','provisional until independent readback and policy acceptance']}\n    dst.parent.mkdir(parents=True,exist_ok=True)\n    with dst.open('x',encoding='utf-8') as f:\n       json.dump(result,f,indent=2,sort_keys=True);f.write('\\n');f.flush();os.fsync(f.fileno())\n    print('T4_CAPACITY_REPORT='+str(dst))\n    print('T4_CAPACITY_RESULT='+result['status'])\n    # No scientifically authorized execution in this script, even if PASS.\n    if not selected:raise SystemExit(2)\n\nif __name__=='__main__':main()\n","t4_eval.py":"#!/usr/bin/env python3\n\"\"\"TTM T4 H1 prereg: train-fitted unigram/bigram; VAL-only story bootstrap.\n\nNo TEST read, no fitting on VALIDATION, no hyperparam selection from TEST.\n\"\"\"\nfrom __future__ import annotations\nimport hashlib, json\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\n\nVOCAB=2048\nALPHA=.1\nBOOT_SEED=20261109\nN_BOOT=1000\n\n\ndef load_private_pack(data_path,manifest_path):\n    m=json.loads(Path(manifest_path).read_text('utf-8'))\n    if m['schema']!='ttm.t4.private.train_val.windowpack.v1' or m['contains_test'] or not m['contains_reversible_train_val_tokens']:\n        raise RuntimeError('PRIVATE_PACK_SCHEMA_OR_TEST_VIOLATION')\n    h=hashlib.sha256()\n    with open(data_path,'rb') as f:\n        for b in iter(lambda:f.read(1<<20),b''):h.update(b)\n    if h.hexdigest()!=m['npz_sha256']:raise RuntimeError('TRAIN_VAL_PACK_SHA_MISMATCH')\n    data=np.load(data_path,allow_pickle=False)\n    expected={f'{p}_{s}' for p in ['train','val'] for s in ['x','y','count','story_index']}\n    if set(data.files)!=expected:raise RuntimeError('UNEXPECTED_DATASET_KEYS_TEST_EXPOSURE')\n    for prefix in ['train','val']:\n        x,y,c,si=[data[f'{prefix}_{suffix}'] for suffix in ['x','y','count','story_index']]\n        if x.shape!=y.shape or x.ndim!=2 or x.shape[1]!=256 or c.shape!=(len(x),) or si.shape!=(len(x),):\n            raise RuntimeError('WINDOW_PACK_SHAPE_INVALID')\n        mask=np.arange(256)[None,:]<c[:,None]\n        if not(np.all(y[~mask]==-100) and np.all((y[mask]>=0)&(y[mask]<VOCAB)) and np.all((x>=0)&(x<VOCAB))):\n            raise RuntimeError('PRIVATE_PACK_MASK_OR_TOKEN_INVALID')\n        expected_count=m['counts']['train_next_token_targets_unique' if prefix=='train' else 'val_next_token_targets']\n        if int(c.sum())!=expected_count:raise RuntimeError('TARGET_COUNT_MANIFEST_MISMATCH')\n        if np.max(si)+1!=m['counts'][f'{prefix}_stories']:raise RuntimeError('STORY_IDS_OR_COUNT_INVALID')\n    return data,m\n\n\ndef fit_baselines(train_x,train_y):\n    # Every source TRAIN pair is counted once; sampling frequency of model training is not part of baseline.\n    mask=train_y!=-100\n    prev=train_x[mask].astype(np.int64)\n    nxt=train_y[mask].astype(np.int64)\n    counts=np.zeros((VOCAB,VOCAB),dtype=np.int64)\n    np.add.at(counts,(prev,nxt),1)\n    unigrams=counts.sum(axis=0).astype(np.float64)\n    smooth_unigrams=(unigrams+ALPHA)/(float(unigrams.sum())+ALPHA*VOCAB)\n    denom=counts.sum(axis=1).astype(np.float64)+ALPHA*VOCAB\n    logbig=np.log((counts.astype(np.float64)+ALPHA)/denom[:,None])\n    loguni=np.log(smooth_unigrams)\n    return loguni,logbig\n\n\ndef baseline_story_scores(data,loguni,logbig):\n    x,y,c,si=[data['val_'+z] for z in ['x','y','count','story_index']]\n    n=int(np.max(si))+1\n    target_counts=np.zeros(n,dtype=np.int64)\n    uni_sum=np.zeros(n,dtype=np.float64);bi_sum=np.zeros(n,dtype=np.float64)\n    for j in range(len(x)):\n        k=int(c[j]);s=int(si[j]);prev=x[j,:k].astype(np.int64);nxt=y[j,:k].astype(np.int64)\n        target_counts[s]+=k\n        uni_sum[s]-=np.sum(loguni[nxt])\n        bi_sum[s]-=np.sum(logbig[prev,nxt])\n    if np.any(target_counts==0):raise RuntimeError('VAL_STORY_WITH_NO_SCORE')\n    return target_counts,uni_sum,bi_sum\n\n@torch.no_grad()\ndef model_story_scores(model,data,device,eval_batch=4):\n    x,y,c,si=[data['val_'+z] for z in ['x','y','count','story_index']]\n    n=int(np.max(si))+1\n    result=np.zeros(n,dtype=np.float64)\n    old=model.training\n    model.eval()\n    try:\n        for a in range(0,len(x),eval_batch):\n            b=min(a+eval_batch,len(x))\n            xx=torch.as_tensor(x[a:b].astype(np.int64),device=device)\n            yy=torch.as_tensor(y[a:b].astype(np.int64),device=device)\n            logits,_=model(xx)\n            loss=F.cross_entropy(logits.float().reshape(-1,VOCAB), yy.reshape(-1),ignore_index=-100,reduction='none')\n            loss=loss.reshape(b-a,256).sum(1).double().cpu().numpy()\n            for j,k in enumerate(range(a,b)):result[int(si[k])]+=float(loss[j])\n    finally:\n        model.train(old)\n    return result\n\n\ndef decide_h1(model_sums,bigram_sums,token_counts,*,seed=BOOT_SEED,boot=N_BOOT):\n    model_sums=np.asarray(model_sums,dtype=np.float64)\n    bigram_sums=np.asarray(bigram_sums,dtype=np.float64)\n    token_counts=np.asarray(token_counts,dtype=np.int64)\n    if not(len(model_sums)==len(bigram_sums)==len(token_counts)) or np.any(token_counts<=0):\n        raise RuntimeError('BOOTSTRAP_STORY_COUNT_INVALID')\n    mnll=model_sums.sum()/token_counts.sum();bnll=bigram_sums.sum()/token_counts.sum()\n    effect=(bnll-mnll)/bnll\n    rng=np.random.default_rng(seed)\n    gaps=[]\n    for _ in range(boot):\n        take=rng.integers(0,len(token_counts),size=len(token_counts),dtype=np.int64)\n        gaps.append(float((bigram_sums[take].sum()-model_sums[take].sum())/token_counts[take].sum()))\n    lo,hi=[float(x) for x in np.quantile(gaps,[.025,.975])]\n    return {'schema':'ttm.t4.h1.val.storybootstrap.v1',\n       'primary_model_val_nll':float(mnll),'primary_bigram_val_nll':float(bnll),\n       'relative_nll_improvement':float(effect),'absolute_nll_gain':float(bnll-mnll),\n       'bootstrap_95pct_gain_ci':[lo,hi],'bootstrap_stories':len(token_counts),\n       'bootstrap_rounds':boot,'bootstrap_seed':seed,'alpha':ALPHA,\n       'scored_val_targets':int(token_counts.sum()),\n       'h1_pass':bool(effect>=.05 and lo>0),\n       'warning':'VALIDATION-only technical computation; must be frozen before T4; TEST sealed'}\n","t4_capacity_accept_a2.py":"#!/usr/bin/env python3\n\"\"\"T4 A2 hardware/memory review from real TRAIN shapes; no arbitrary wall-time cap.\n\nNo model training, no TEST, no science permission. Pair the synthetic GPU bench\nwith TRUE number of valid token targets/window; never claim a budget from padded slots.\n\"\"\"\nfrom __future__ import annotations\nimport argparse,hashlib,json,math,os\nfrom pathlib import Path\n\n\ndef file_sha(p):\n h=hashlib.sha256()\n with open(p,'rb') as f:\n  for b in iter(lambda:f.read(1<<20),b''):h.update(b)\n return h.hexdigest()\n\n\ndef assess(cap,pack):\n if cap['schema']!='ttm.t4.capacity.synthetic.v1' or cap['science_executed'] is not False:\n  raise RuntimeError('CAPACITY_REPORT_INVALID')\n if pack['schema']!='ttm.t4.private.train_val.windowpack.v1' or pack['contains_test']:\n  raise RuntimeError('PACK_SCOPE_OR_TEST_INVALID')\n mean=pack['counts']['train_mean_valid_targets_per_window']\n if not 1<=mean<=256:raise RuntimeError('TRAIN_WINDOW_MEAN_INVALID')\n inflation=max(1.0,256/mean)\n # Extra guard on top of synthetic bench 25% capacity reserve.\n risk_factor=inflation*1.10\n rows=[]\n for r in cap['records']:\n  projected=(r['projected_science_seconds_with_25pct_reserved']*risk_factor)\n  rows.append({'precision':r['precision'],'microbatch_windows':r['microbatch_windows'],\n        'gradient_accumulation':r['gradient_accumulation'],\n        'synthetic_worst_seconds':r['worst_seconds'],\n        'actual_window_inflation':inflation,'extra_safety_factor':1.10,\n        'projected_seconds_conservative':round(projected,1),\n        'vram_reserved_ratio':r['memory_reserved_ratio'],\n        'fits_vram_for_free_tier_trial':bool(r['memory_reserved_ratio']<=.8)})\n eligible=[x for x in rows if x['fits_vram_for_free_tier_trial']]\n selected=min(eligible,key=lambda x:(x['projected_seconds_conservative'],x['vram_reserved_ratio'])) if eligible else None\n return {'schema':'ttm.t4.capacity.privatepack.review.v1','status':'CAPACITY_PASS_PROVISIONAL_FOR_SCIENCE_REVIEW' if selected else 'CAPACITY_HOLD',\n  'science_executed':False,'no_test_used':True,'source_capacity_status':cap['status'],\n  'train_scored_targets_unique':pack['counts']['train_next_token_targets_unique'],\n  'effective_train_target_reuse_factor':pack['counts']['train_reuse_factor_for_4194304_target_budget'],\n  'mean_train_valid_targets_per_window':mean,\n  'gpu_time_policy':'COLAB_FREE_RUN_UNTIL_1024_OR_PREEMPTED',\n  'no_hard_elapsed_limit':True,\n  'safety_risk_factor':risk_factor,\n  'candidates':rows,'selected':selected,\n  'non_proven':['No fixed 55/90 minute hard-stop; Colab Free runtime may still preempt',\n   'Full 1024-step real-science runtime not executed',\n   'No science permission from this reviewer']}\n\n\ndef main():\n p=argparse.ArgumentParser();p.add_argument('--capacity',required=True);p.add_argument('--pack-manifest',required=True);p.add_argument('--out',required=True)\n a=p.parse_args()\n out=Path(a.out)\n if out.exists():raise FileExistsError('CAPACITY_REVIEW_ALREADY_EXISTS')\n cap=json.loads(Path(a.capacity).read_text('utf8'));pack=json.loads(Path(a.pack_manifest).read_text('utf8'))\n result=assess(cap,pack)\n result['capacity_sha256']=file_sha(a.capacity);result['pack_manifest_sha256']=file_sha(a.pack_manifest)\n with out.open('x',encoding='utf8') as f:json.dump(result,f,indent=2,sort_keys=True);f.write('\\n');f.flush();os.fsync(f.fileno())\n print('T4_CAPACITY_PRIVATE_PACK_REVIEW='+result['status'])\n print('T4_SCIENCE_LOCK=HOLD')\n\nif __name__=='__main__':main()\n","t4_train_science_a2.py":"#!/usr/bin/env python3\n\"\"\"TTM T4 A2 continuous 1024-step science trainer. Exact hash-pinned one-shot lock.\n\nThis code is not permission to start training. No TEST read. 4096 real targets/step,\n1024 steps, frozen optimizer, VAL NLL baseline, READY-last immutable checkpoints.\n\"\"\"\nfrom __future__ import annotations\nimport argparse, hashlib, importlib.util, json, math, os, random, shutil, sys, time\nfrom pathlib import Path\nimport numpy as np\nimport torch\nfrom torch.nn import functional as F\nfrom t4_capacity_smoke import P02_BLOB,CONFIG,TOTAL_STEPS,TOKENS_PER_STEP,p02_module,SEED\nfrom t4_eval import load_private_pack,fit_baselines,baseline_story_scores,model_story_scores,decide_h1\n\nCHECKPOINT_STEPS=(256,512,768,1024)\nVALID_STEPS=tuple(range(128,1025,128))\nTIME_POLICY='COLAB_FREE_RUN_UNTIL_1024_OR_PREEMPTED'  # No arbitrary 55/90-minute hard stop\n\n\ndef sha_file(p):\n    h=hashlib.sha256()\n    with open(p,'rb') as f:\n        for b in iter(lambda:f.read(1<<20),b''):h.update(b)\n    return h.hexdigest()\n\n\ndef json_new(p,obj):\n    with Path(p).open('x',encoding='utf-8') as f:\n        json.dump(obj,f,indent=2,sort_keys=True);f.write('\\n');f.flush();os.fsync(f.fileno())\n\n\ndef require_science_lock(path,expected_sha,pack_manifest,capacity_sha,capacity_review_sha):\n    if not path or not expected_sha or len(expected_sha)!=64:\n        raise RuntimeError('SCIENCE_NOT_AUTHORIZED_NO_EXACT_LOCK_SHA')\n    if sha_file(path)!=expected_sha:raise RuntimeError('SCIENCE_LOCK_SHA_MISMATCH')\n    obj=json.loads(Path(path).read_text('utf8'))\n    if obj.get('authorization')!='ONE_SHOT_T4_SCIENCE_GRANTED' or obj.get('science_seed')!=SEED:\n        raise RuntimeError('SCIENCE_AUTHORIZATION_NOT_GRANTED')\n    if obj.get('budget')!={'optimizer_steps':1024,'target_tokens_per_step':4096,'time_policy':TIME_POLICY}:\n        raise RuntimeError('SCIENCE_BUDGET_DIFFERENT')\n    if obj.get('train_pack_sha256')!=pack_manifest['npz_sha256'] or obj.get('capacity_report_sha256')!=capacity_sha or obj.get('capacity_review_sha256')!=capacity_review_sha:\n        raise RuntimeError('SCIENCE_DATA_OR_CAPACITY_SHA_MISMATCH')\n    if obj.get('precision') not in ('fp32','fp16') or obj.get('microbatch_windows') not in (1,2,4,8):\n        raise RuntimeError('PRECISION_OR_MICROBATCH_NOT_FROZEN')\n    if obj.get('trainer_a2_sha256')!=sha_file(__file__):\n        raise RuntimeError('T4_A2_TRAINER_HASH_CHANGED')\n    if obj.get('p02_git_blob')!=P02_BLOB:\n        raise RuntimeError('MODEL_ARCH_CHANGED')\n    return obj\n\n\ndef lambda_schedule(step):\n    # LambdaLR gets epoch index 0 after initialization; 32 linear warmup steps.\n    if step<32:return (step+1)/32\n    t=min((step-32)/(TOTAL_STEPS-32),1.0)\n    return .1+.9*(.5+.5*math.cos(math.pi*t))\n\n\ndef model_context(p02,device,precision):\n    torch.manual_seed(SEED);random.seed(SEED);np.random.seed(SEED)\n    torch.cuda.manual_seed_all(SEED)\n    model=p02.GPT(p02.P02Config(**CONFIG)).to(device)\n    if sum(p.numel() for p in model.parameters())!=3749376:raise RuntimeError('P02_ARCH_CHANGED')\n    optimizer=torch.optim.AdamW(model.parameters(),lr=3e-4,betas=(.9,.95),eps=1e-8,weight_decay=.1)\n    scheduler=torch.optim.lr_scheduler.LambdaLR(optimizer,lr_lambda=lambda_schedule)\n    sampler=torch.Generator(device='cpu').manual_seed(SEED+71)\n    scaler=torch.amp.GradScaler('cuda') if precision=='fp16' else None\n    return model,optimizer,scheduler,sampler,scaler\n\n\ndef sample_exact_step(data,sampler,microbatch):\n    \"\"\"Draw train-only windows until EXACTLY 4096 valid target labels are present.\"\"\"\n    x=data['train_x'];y=data['train_y'];c=data['train_count']\n    examples=[];total=0\n    while total<TOKENS_PER_STEP:\n        k=int(torch.randint(len(x),(1,),generator=sampler).item())\n        n=min(int(c[k]),TOKENS_PER_STEP-total)\n        if n<=0:raise RuntimeError('ZERO_TARGET_WINDOW')\n        yy=y[k].copy()\n        yy[n:]=-100\n        examples.append((x[k],yy))\n        total+=n\n    if total!=TOKENS_PER_STEP:raise RuntimeError('BAD_EXACT_TARGET_BUDGET')\n    groups=[]\n    for st in range(0,len(examples),microbatch):\n        sub=examples[st:st+microbatch]\n        groups.append((np.stack([i[0] for i in sub]),np.stack([i[1] for i in sub])))\n    return groups,len(examples)\n\n\ndef train_one_step(model,opt,sch,sampler,scaler,data,device,precision,microbatch):\n    model.train()\n    opt.zero_grad(set_to_none=True)\n    groups,windows=sample_exact_step(data,sampler,microbatch)\n    total_loss=0.0\n    for x,y in groups:\n        xx=torch.tensor(x,dtype=torch.long,device=device)\n        yy=torch.tensor(y,dtype=torch.long,device=device)\n        with torch.autocast(device_type='cuda',dtype=torch.float16,enabled=(precision=='fp16')):\n            logits,_=model(xx)\n            loss=F.cross_entropy(logits.float().reshape(-1,2048),yy.reshape(-1),ignore_index=-100,reduction='sum')/TOKENS_PER_STEP\n        if not torch.isfinite(loss).item():raise RuntimeError('T4_NONFINITE_LOSS')\n        if scaler:scaler.scale(loss).backward()\n        else:loss.backward()\n        total_loss+=float(loss.detach().cpu())\n    if scaler:\n        scaler.unscale_(opt)\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)\n        old=scaler.get_scale();scaler.step(opt);scaler.update()\n        if scaler.get_scale()<old:raise RuntimeError('T4_FP16_OVERFLOW_STEP_SKIPPED')\n    else:\n        torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)\n        opt.step()\n    sch.step()\n    return total_loss,windows\n\n\ndef _numpy_rng_to_primitives():\n    a=np.random.get_state()\n    return (a[0],a[1].tolist(),int(a[2]),int(a[3]),float(a[4]))\n\n\ndef checkpoint_payload(model,opt,sch,sampler,scaler,step,lock_sha,pack_sha,elapsed,device):\n    return {'schema':'ttm.t4.science.checkpoint.v1','step':step,'science_seed':SEED,\n      'p02_git_blob':P02_BLOB,'config':CONFIG,'science_lock_sha256':lock_sha,\n      'train_pack_sha256':pack_sha,'elapsed_gpu_seconds_total':elapsed,\n      'model':model.state_dict(),'optimizer':opt.state_dict(),\n      'scheduler':sch.state_dict(),'sampler_state':sampler.get_state(),\n      'torch_cpu_rng':torch.get_rng_state(),'torch_cuda_rng':torch.cuda.get_rng_state_all(),\n      'python_rng':random.getstate(),'numpy_rng':_numpy_rng_to_primitives(),\n      'grad_scaler_state':scaler.state_dict() if scaler else None,'device_type':device.type,\n      'torch_version':str(torch.__version__)}\n\n\ndef checkpoint_write(run_root,step,payload,log,local_scratch):\n    \"\"\"Write locally, copy + SHA readback to mounted Drive, READY **last**.\"\"\"\n    folder=run_root/'checkpoints'/f'step_{step:04d}'\n    if folder.exists():raise FileExistsError('T4_CHECKPOINT_STEP_ALREADY_EXISTS')\n    folder.mkdir(parents=True,exist_ok=False)\n    local_scratch.mkdir(parents=True,exist_ok=True)\n    tmp=local_scratch/f'ckpt_{step:04d}.pt'\n    if tmp.exists():raise FileExistsError('T4_LOCAL_CHECKPOINT_ALREADY_EXISTS')\n    torch.save(payload,tmp)\n    hs=sha_file(tmp);n=tmp.stat().st_size\n    dst=folder/'checkpoint.pt'\n    with tmp.open('rb') as src,dst.open('xb') as dest:\n        shutil.copyfileobj(src,dest,length=1<<20);dest.flush();os.fsync(dest.fileno())\n    if sha_file(dst)!=hs or dst.stat().st_size!=n:raise RuntimeError('T4_DRIVE_CHECKPOINT_SHA_FAILED')\n    log_dst=folder/'progress.json'\n    json_new(log_dst,{'schema':'ttm.t4.progress.v1','step':step,'records':log})\n    ready={'schema':'ttm.t4.ready.last.v1','step':step,'checkpoint':'checkpoint.pt',\n           'bytes':n,'sha256':hs,'science_lock_sha256':payload['science_lock_sha256'],\n           'train_pack_sha256':payload['train_pack_sha256'],\n           'progress_sha256':sha_file(log_dst)}\n    json_new(folder/'READY.json',ready)\n    # Local scratch is disposable; leave original on partial failure for investigation.\n    return ready\n\n\ndef checkpoint_load(folder,model,opt,sch,sampler,scaler,device,lock_sha,pack_sha):\n    ready=json.loads((folder/'READY.json').read_text('utf8'))\n    cp=folder/ready['checkpoint']\n    if sha_file(cp)!=ready['sha256'] or cp.stat().st_size!=ready['bytes']:\n        raise RuntimeError('T4_CHECKPOINT_DURABILITY_OR_SHA_MISMATCH')\n    payload=torch.load(cp,map_location='cpu',weights_only=True)\n    if payload['schema']!='ttm.t4.science.checkpoint.v1' or payload['p02_git_blob']!=P02_BLOB or payload['science_lock_sha256']!=lock_sha or payload['train_pack_sha256']!=pack_sha:\n        raise RuntimeError('T4_RESUME_MANIFEST_PIN_MISMATCH')\n    model.load_state_dict(payload['model'],strict=True)\n    opt.load_state_dict(payload['optimizer']);sch.load_state_dict(payload['scheduler'])\n    sampler.set_state(payload['sampler_state'].cpu())\n    torch.set_rng_state(payload['torch_cpu_rng'].cpu())\n    if device.type=='cuda':torch.cuda.set_rng_state_all([a.cpu() for a in payload['torch_cuda_rng']])\n    random.setstate(tuple(payload['python_rng']))\n    nr=payload['numpy_rng']\n    np.random.set_state((nr[0],np.asarray(nr[1],dtype=np.uint32),nr[2],nr[3],nr[4]))\n    if scaler:\n        if payload['grad_scaler_state'] is None:raise RuntimeError('T4_GRADSCALER_NOT_SAVED')\n        scaler.load_state_dict(payload['grad_scaler_state'])\n    return payload['step'],float(payload['elapsed_gpu_seconds_total'])\n\n\ndef main():\n    ap=argparse.ArgumentParser()\n    ap.add_argument('--mode',choices=['preflight','science'],default='preflight')\n    ap.add_argument('--p02',required=True)\n    ap.add_argument('--data',required=True);ap.add_argument('--data-manifest',required=True)\n    ap.add_argument('--capacity-report',required=True)\n    ap.add_argument('--capacity-review',required=True)\n    ap.add_argument('--drive-run',required=True)\n    ap.add_argument('--lock-json');ap.add_argument('--lock-sha256')\n    ap.add_argument('--resume-checkpoint-dir')\n    args=ap.parse_args()\n    if args.mode!='science':\n        print('TTM_T4_TRAINING=NOT_AUTHORIZED_PREFLIGHT_ONLY');print('SCIENCE_LOCK=HOLD');return\n    # No operation on GPU or data until complete exact signed-off lock readback.\n    data_manifest=json.loads(Path(args.data_manifest).read_text('utf8'))\n    cap=json.loads(Path(args.capacity_report).read_text('utf8'))\n    cap_sha=sha_file(args.capacity_report)\n    review=json.loads(Path(args.capacity_review).read_text('utf8'))\n    review_sha=sha_file(args.capacity_review)\n    if review.get('status')!='CAPACITY_PASS_PROVISIONAL_FOR_SCIENCE_REVIEW' or review.get('capacity_sha256')!=cap_sha or review.get('science_executed') is not False:\n        raise RuntimeError('CAPACITY_PRIVATE_PACK_REVIEW_NOT_ACCEPTED')\n    if cap.get('status')!='PASS_PROVISIONAL_CAPACITY_ONLY' or cap.get('science_executed') is not False:\n        raise RuntimeError('CAPACITY_NOT_ACCEPTABLE_FOR_SCIENCE')\n    lock=require_science_lock(args.lock_json,args.lock_sha256,data_manifest,cap_sha,review_sha)\n    selected=review.get('selected') or {}\n    if lock['precision']!=selected.get('precision') or lock['microbatch_windows']!=selected.get('microbatch_windows'):\n        raise RuntimeError('LOCKED_BATCH_OR_PRECISION_DIFFERS_FROM_MEASURED_CAPACITY')\n    if Path(args.drive_run).exists() and not args.resume_checkpoint_dir:\n        raise FileExistsError('T4_ONE_SHOT_RUN_ID_ALREADY_EXISTS')\n    if not torch.cuda.is_available():raise RuntimeError('T4_SCIENCE_REQUIRES_GPU')\n    p02=p02_module(args.p02)\n    data,manifest=load_private_pack(args.data,args.data_manifest)\n    if manifest['npz_sha256']!=lock['train_pack_sha256']:raise RuntimeError('LOCKED_DATA_INVALID')\n    model,opt,sch,sampler,scaler=model_context(p02,torch.device('cuda'),lock['precision'])\n    out=Path(args.drive_run)\n    if not args.resume_checkpoint_dir:out.mkdir(parents=True,exist_ok=False)\n    else:\n        if not out.is_dir():raise RuntimeError('MISSING_CONTINUATION_RUN_DIR')\n    if args.resume_checkpoint_dir:\n        start,spent=checkpoint_load(Path(args.resume_checkpoint_dir),model,opt,sch,sampler,scaler,\n            torch.device('cuda'),args.lock_sha256,manifest['npz_sha256'])\n        prev=Path(args.resume_checkpoint_dir)/'progress.json'\n        log=json.loads(prev.read_text('utf8'))['records']\n    else:start=0;spent=0.0;log=[]\n    if start>=TOTAL_STEPS:\n        raise RuntimeError('T4_ALREADY_REACHED_1024_NO_RERUN')\n    print('T4_SCIENCE_1024_CONTINUOUS_START='+str(start), flush=True)\n    print('T4_TIME_POLICY='+TIME_POLICY, flush=True)\n    loguni,logbi=fit_baselines(data['train_x'],data['train_y'])\n    counts,unis,bis=baseline_story_scores(data,loguni,logbi)\n    for step in range(start+1,TOTAL_STEPS+1):\n        torch.cuda.synchronize();t0=time.perf_counter()\n        loss,drawn=train_one_step(model,opt,sch,sampler,scaler,data,torch.device('cuda'),lock['precision'],lock['microbatch_windows'])\n        entry={'step':step,'train_loss':loss,'windows_drawn':drawn,'exact_scored_targets':4096}\n        if step in VALID_STEPS:\n            model_sums=model_story_scores(model,data,torch.device('cuda'))\n            entry['val']=decide_h1(model_sums,bis,counts)\n            entry['val_unigram_nll']=float(unis.sum()/counts.sum())\n        torch.cuda.synchronize();spent+=time.perf_counter()-t0\n        log.append(entry)\n        if step in CHECKPOINT_STEPS:\n            payload=checkpoint_payload(model,opt,sch,sampler,scaler,step,args.lock_sha256,manifest['npz_sha256'],spent,torch.device('cuda'))\n            ready=checkpoint_write(out,step,payload,log,Path('/content/ttm_t4_ckpt_local'))\n            print('T4_CHECKPOINT_READY_STEP='+str(step)+' SHA256='+ready['sha256'],flush=True)\n        if step==start+1 or step%32==0 or step==TOTAL_STEPS:\n            print('T4_STEP='+str(step)+'/'+str(TOTAL_STEPS)+' TRAIN_LOSS='+format(loss,'.6f')+' GPU_COMPUTE_S='+format(spent,'.1f'),flush=True)\n    final={'schema':'ttm.t4.science.result.v1','status':'COMPLETED_REQUIRES_INDEPENDENT_ADJUDICATION',\n       'last_step':TOTAL_STEPS,'elapsed_gpu_seconds':spent,'h1_val':log[-1]['val'],\n       'no_test_used':True,'exact_scored_training_tokens':TOTAL_STEPS*TOKENS_PER_STEP,\n       'time_policy':TIME_POLICY,'no_gpu_wall_clock_hard_cap':True,\n       'science_lock_sha256':args.lock_sha256,'data_pack_sha256':manifest['npz_sha256']}\n    json_new(out/'T4_FINAL_UNADJUDICATED.json',final)\n    print('TTM_T4_FINISHED_UNADJUDICATED='+str(out/'T4_FINAL_UNADJUDICATED.json'))\n\nif __name__=='__main__':main()\n"}
def sha256_file(p):
    h=hashlib.sha256()
    with Path(p).open('rb') as f:
        for block in iter(lambda:f.read(1<<20),b''): h.update(block)
    return h.hexdigest()
for name,(rel,expected) in FILES.items():
    body=EMBEDDED[name].encode('utf-8')
    assert hashlib.sha256(body).hexdigest()==expected,'SOURCE_EMBEDDED_SHA_MISMATCH:'+name
    dst=ROOT/name
    if dst.exists():
        assert sha256_file(dst)==expected,'SOURCE_EXISTING_MISMATCH:'+name
    else:
        with dst.open('xb') as f: f.write(body)
    print('T4_A2_SOURCE_PIN=PASS',name)
sys.path.insert(0,str(ROOT))
print('T4_A2_SOURCE_COMMIT='+SOURCE_COMMIT)



In [ ]:
# Ô 2 — xác minh chính xác TRAIN/VAL riêng tư, tạo lock đúng một lượt, không dùng TEST
from google.colab import drive
from pathlib import Path
import os, json, hashlib, torch, shutil
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'COLAB_FREE_GPU_REQUIRED'
print('T4_GPU='+torch.cuda.get_device_name(0), 'TORCH='+str(torch.__version__))
BASE=Path('/content/drive/MyDrive/TTM/ttm-small-pilot')
PRIVATE=BASE/'t4_private_pack_20261108_v1'
CAPACITY=BASE/'t4_capacity_synthetic_20261108_v1'/'T4_CAPACITY_REPORT.json'
SRC=PRIVATE/'T4_PRIVATE_TRAIN_VAL_WINDOWS.npz'
MNF=PRIVATE/'T4_PRIVATE_PACK_MANIFEST.json'
RUN=BASE/'t4_science_4m_1024_a2_20261108_v1'
DATA_SHA='184f68b4c0c800ef3e98362c2e20e1cd35baa269e926489a475664526ecd9913'
MANIFEST_SHA='ef2e6e67d49b1d5a921b1051df329f15574f1c10cece0ce49468acb53322d650'
CAPACITY_SHA='c55a568496a888bc3abc95c0231a5ed17d3a6f366361571b6f89a4457d09f1ee'
assert SRC.is_file() and MNF.is_file() and CAPACITY.is_file(), 'MISSING_PINNED_DATA_OR_CAPACITY'
assert SRC.stat().st_size==1186107 and sha256_file(SRC)==DATA_SHA,'PRIVATE_NPZ_SHA_FAILED'
assert sha256_file(MNF)==MANIFEST_SHA,'PRIVATE_MANIFEST_SHA_FAILED'
assert sha256_file(CAPACITY)==CAPACITY_SHA,'CAPACITY_SHA_FAILED'
manifest=json.loads(MNF.read_text('utf8'))
assert manifest['npz_sha256']==DATA_SHA and manifest['contains_test'] is False
assert manifest['counts']['train_stories']==2048 and manifest['counts']['val_stories']==256
assert manifest['counts']['test_stories_tokenized']==0
assert manifest['counts']['train_next_token_targets_unique']==498287
capacity=json.loads(CAPACITY.read_text('utf8'))
assert capacity['science_executed'] is False and capacity['status']=='PASS_PROVISIONAL_CAPACITY_ONLY'
from t4_capacity_accept_a2 import assess
review=assess(capacity,manifest)
review['capacity_sha256']=CAPACITY_SHA
review['pack_manifest_sha256']=MANIFEST_SHA
selected=review['selected']
assert review['status']=='CAPACITY_PASS_PROVISIONAL_FOR_SCIENCE_REVIEW'
assert selected['precision']=='fp16' and selected['microbatch_windows']==8
def write_exact_once(path,obj):
    raw=(json.dumps(obj,indent=2,sort_keys=True)+'\n').encode('utf-8')
    if path.exists():
        assert path.read_bytes()==raw, 'LOCK_OR_REVIEW_CONFLICT:'+path.name
    else:
        with path.open('xb') as f:
            f.write(raw);f.flush();os.fsync(f.fileno())
    return sha256_file(path)
REVIEW=PRIVATE/'T4_CAPACITY_PRIVATE_PACK_REVIEW_A2.json'
review_sha=write_exact_once(REVIEW,review)
LOCK=PRIVATE/'T4_SCIENCE_LOCK_A2_1024.json'
LOCK_OBJ={
 'schema':'ttm.t4.a2.science.lock.v1',
 'authorization':'ONE_SHOT_T4_SCIENCE_GRANTED',
 'science_seed':20261108,
 'budget':{'optimizer_steps':1024,'target_tokens_per_step':4096,
           'time_policy':'COLAB_FREE_RUN_UNTIL_1024_OR_PREEMPTED'},
 'train_pack_sha256':DATA_SHA,
 'capacity_report_sha256':CAPACITY_SHA,
 'capacity_review_sha256':review_sha,
 'precision':'fp16','microbatch_windows':8,
 'p02_git_blob':'9762343e06cf800b50574c2871ae1ed58537fb18',
 'trainer_a2_sha256':FILES['t4_train_science_a2.py'][1],
 'run_id':RUN.name,'no_test_access':True,
 'h1_primary_checkpoint_step':1024}
LOCK_SHA=write_exact_once(LOCK,LOCK_OBJ)
print('T4_PRIVATE_DRIVE_SHA=PASS_EXACT')
print('T4_SCIENCE_LOCK_SHA256='+LOCK_SHA)
print('T4_PLAN=CONTINUOUS_1024_NO_STOP_AT_256')


In [ ]:
# Ô 3 — chạy một lượt đến 1.024; tự checkpoint, resume khi cần và luôn flush Drive
import subprocess, sys, json, shutil, os
local_npz=ROOT/'T4_PRIVATE_TRAIN_VAL_WINDOWS.npz'
local_manifest=ROOT/'T4_PRIVATE_PACK_MANIFEST.json'
if local_npz.exists(): assert sha256_file(local_npz)==DATA_SHA,'LOCAL_DATA_MISMATCH'
else: shutil.copyfile(SRC,local_npz)
if local_manifest.exists(): assert sha256_file(local_manifest)==MANIFEST_SHA,'LOCAL_MANIFEST_MISMATCH'
else: shutil.copyfile(MNF,local_manifest)
assert sha256_file(local_npz)==DATA_SHA and sha256_file(local_manifest)==MANIFEST_SHA
assert sha256_file(LOCK)==LOCK_SHA and sha256_file(REVIEW)==review_sha
resume=None
completed=False
if RUN.exists():
    if (RUN/'T4_FINAL_UNADJUDICATED.json').is_file():
        completed=True
        print('T4_ALREADY_COMPLETED_1024=NO_RERUN')
    else:
        croot=RUN/'checkpoints'
        folders=sorted(croot.glob('step_*')) if croot.exists() else []
        if not folders: raise RuntimeError('RUN_EXISTS_WITHOUT_READY_CHECKPOINT_STOP')
        if any(not (f/'READY.json').is_file() for f in folders):
            raise RuntimeError('PARTIAL_CHECKPOINT_NEEDS_REVIEW_NO_AUTO_RERUN')
        resume=folders[-1]
        assert resume.name in ('step_0256','step_0512','step_0768'), 'UNEXPECTED_RESUME_STEP'
        print('T4_RESUME_SAME_RUN_FROM='+resume.name)
if not completed:
    args=[sys.executable,'-u',str(ROOT/'t4_train_science_a2.py'),'--mode','science',
      '--p02',str(ROOT/'p02_source_exact.py'),
      '--data',str(local_npz),'--data-manifest',str(local_manifest),
      '--capacity-report',str(CAPACITY),'--capacity-review',str(REVIEW),
      '--drive-run',str(RUN),'--lock-json',str(LOCK),'--lock-sha256',LOCK_SHA]
    if resume is not None: args.extend(['--resume-checkpoint-dir',str(resume)])
    try:
        with subprocess.Popen(args,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,
                              text=True,bufsize=1) as proc:
            for line in proc.stdout: print(line,end='',flush=True)
            rc=proc.wait()
        if rc!=0: raise RuntimeError('T4_STOP_KEEP_DRIVE_EVIDENCE_NO_NEW_RUN_EXIT_'+str(rc))
        final=json.loads((RUN/'T4_FINAL_UNADJUDICATED.json').read_text('utf8'))
        assert final['last_step']==1024 and final['exact_scored_training_tokens']==4194304
        print('T4_A2_COMPLETE_1024=PASS_UNADJUDICATED')
        print('T4_A2_H1_VAL_RESULT='+json.dumps(final['h1_val'],sort_keys=True))
    finally:
        try: drive.flush_and_unmount()
        finally: print('T4_DRIVE_FLUSH_ATTEMPTED_READBACK_REQUIRED')
else:
    print('T4_FINAL_REPORT_EXISTS=DO_NOT_RETRAIN')
    drive.flush_and_unmount()
